# Greek Banking Sector — Executive Summary & Investment Implications

**Scope:** Eurobank, Alpha Bank, Piraeus Bank, NBG — FY2022–FY2024  
**Data source:** Audited annual reports (PDFs), validated against `kpis_final.csv` and `greek_banking_final.db`  
**Analyst:** Spyros Papastergiou — spyrossyo96@gmail.com

---

This notebook synthesises findings from notebooks 01–05 into three outputs:
1. A one-page **KPI scorecard** (the recruiter's first stop)
2. **Investment implications** — what a buy-side or sell-side analyst would do with this data
3. A **risk matrix** for the sector heading into 2025–2026

In [1]:
import sqlite3
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

BANKS  = ['Eurobank', 'Alpha Bank', 'Piraeus Bank', 'NBG']
COLORS = {'Eurobank': '#0067B1', 'Alpha Bank': '#E2231A',
           'Piraeus Bank': '#F7A600', 'NBG': '#003087'}

con  = sqlite3.connect('../02_Banking_Sector_Dashboard/data/greek_banking_final.db')
kpis = pd.read_sql('SELECT * FROM kpis ORDER BY bank, year', con)
con.close()

k24 = kpis[kpis.year == 2024].set_index('bank')
print(f'Loaded {len(kpis)} rows — {kpis.bank.nunique()} banks x {kpis.year.nunique()} years')

Loaded 12 rows — 4 banks x 3 years


## 1. One-Page KPI Scorecard (FY2024)

In [2]:
# ── Scorecard table ───────────────────────────────────────────────────────────
cols = ['roe', 'roa', 'nim', 'cost_to_income', 'cet1', 'loan_to_deposit', 'net_profit', 'total_assets']
labels = ['ROE (%)', 'ROA (%)', 'NIM (%)', 'C/I (%)', 'CET1 (%)', 'L/D (%)', 'Net Profit (€m)', 'Assets (€bn)']

score = k24.loc[BANKS, cols].copy()
score['total_assets'] = score['total_assets'] / 1000
score.columns = labels

# Format
fmt = {
    'ROE (%)': '{:.1f}', 'ROA (%)': '{:.2f}', 'NIM (%)': '{:.2f}',
    'C/I (%)': '{:.1f}', 'CET1 (%)': '{:.1f}', 'L/D (%)': '{:.1f}',
    'Net Profit (€m)': '{:,.0f}', 'Assets (€bn)': '{:.1f}'
}

def best(col):
    """Return index of best value per metric (direction-aware)."""
    lower_better = {'C/I (%)': True, 'L/D (%)': True}
    return score[col].idxmin() if lower_better.get(col) else score[col].idxmax()

print('FY2024 PEER SCORECARD')
print('=' * 80)
print(f'{"":20}' + ''.join(f'{b:>15}' for b in BANKS))
print('-' * 80)
for lbl, col in zip(labels, cols):
    row_data = score[lbl]
    winner = best(lbl)
    row_str = f'{lbl:20}'
    for b in BANKS:
        val = fmt[lbl].format(row_data[b])
        flag = ' ★' if b == winner else '  '
        row_str += f'{val + flag:>15}'
    print(row_str)
print('=' * 80)
print('★ = best in class for that metric')

FY2024 PEER SCORECARD
                           Eurobank     Alpha Bank   Piraeus Bank            NBG
--------------------------------------------------------------------------------
ROE (%)                      16.9 ★          8.2           12.9           13.7  
ROA (%)                      1.44           0.94           1.33           1.54 ★
NIM (%)                      2.48           2.32           2.61           3.14 ★
C/I (%)                      32.6           37.7           31.8 ★         36.5  
CET1 (%)                     17.6           15.4           13.7           19.1 ★
L/D (%)                      64.6           76.5           64.7           62.7 ★
Net Profit (€m)             1,458 ★          668          1,066          1,158  
Assets (€bn)                101.2 ★         71.0           80.0           75.0  
★ = best in class for that metric


## 2. Investment Implications

> *"If I were running a Greek-equity sleeve in an EM fund, what would this data tell me?"*

---

### 2.1 The Rate Tailwind Is Peaking — NIM Compression Is the 2025 Theme

All four banks expanded NIM materially from 2022 to 2024, riding the ECB hiking cycle.  
**NBG** reached the highest NIM (3.14%), making it most sensitive to further ECB cuts.  
**Eurobank** delivered the most stable NIM trajectory (1.90% → 2.72% → 2.48%), already showing early compression in 2024 — a leading indicator.

**Implication:** NIM-sensitive revenue will face headwinds in 2025–2026 as the ECB MRO approaches 2.0–2.5%. Banks with higher fee income share (Eurobank: 30%+ of op. income) are better insulated than pure-play NII earners.

---

### 2.2 NBG: Sector Leader on Both ROA and Capital

NBG leads the sector on **ROA (1.54%)** and **CET1 (19.1%)** — a rare combination of high profitability and capital strength.  
CET1 of 19.1% sits ~650bps above the regulatory floor, creating significant capital return optionality:
- Share buybacks / special dividends  
- M&A capacity without equity issuance  
- Buffer to absorb NIM compression without touching payout

NIM at 3.14% (sector-high) makes NBG the most rate-sensitive name — a two-edged sword depending on ECB trajectory.

**Implication:** NBG screens as a *quality + capital return story*. Watch for 2025 payout announcements. For income-oriented mandates, this is the most compelling name in the sector.

---

### 2.3 Eurobank: Best M&A Execution, Strong Efficiency

Eurobank ranks second on C/I (32.6%) and delivered the most complex capital-markets operation of the cycle — the Hellenic Bank consolidation. The deal added ~€20bn in assets (+26.7% YoY) at a badwill gain, accretive day-one.  
CET1 of 17.6% supports progressive capital return.

**Implication:** Eurobank earns a quality premium through M&A track record and fee income diversification. Investors who want Greek bank exposure with lower NIM sensitivity should favour Eurobank over NBG heading into 2025 rate cuts.

---

### 2.4 Piraeus: Lowest C/I, But Earnings Quality Concern

**Piraeus leads the sector on C/I efficiency (31.8%)** — the best cost discipline in the peer group. However, net profit jumped 35.3% in 2024 (€788m → €1,066m) largely due to impairment release (€523m → €181m — a 65% drop in one year). Strip out the impairment benefit and underlying profit growth is closer to flat.

**Implication:** Piraeus 2024 ROE (12.9%) flatters underlying profitability. A normalisation of credit costs toward sector average (~€300–400m) would reduce net profit by ~15–20%. **Lowest CET1 (13.7%)** also limits capital return potential until the buffer rebuilds to at least 14–15%. Attractive on efficiency metrics, but approach with caution on earnings quality.

---

### 2.5 Alpha Bank: Cost Efficiency Is the Structural Lag

Alpha's C/I has improved dramatically (49.0% → 37.7%), but it still trails Piraeus and Eurobank. The gap is structural: Alpha's operating expenses fell 20% from 2022 to 2023 but partially reversed in 2024 as the bank invested in digital transformation.

**Implication:** The cost-reduction story is real but not yet complete. If Alpha reaches a 33–34% C/I by 2026, earnings uplift would be material — approximately €200m in additional pre-tax profit at current revenues.

---

### 2.6 Sector-Level Takeaway

Greek banks entered 2025 in their strongest position since the pre-GFC era: NPL ratios in mid-single-digits (vs. 45%+ at peak), CET1 above 13% across the board, and ROE recovering toward double digits for all four.  
The key risk to this thesis is **ECB rate cuts + NIM compression** outpacing banks' ability to grow fee income and loan volumes.

**If I were building a Greek-bank exposure today:** Overweight NBG (best ROA + capital optionality), hold Eurobank (quality + M&A execution + NIM resilience), underweight Piraeus (earnings quality concern until credit cost normalises).

## 3. NIM Compression Scenario (2025–2026)

In [3]:
# Simple NIM compression sensitivity: assume 50% pass-through of ECB rate change to NIM
# ECB MRO: 2024 base 4.00%, base-case 2025 = 2.50%, bear = 2.00%
ecb = {'2024A': 4.00, 'Base 2025E': 2.50, 'Bear 2025E': 2.00, 'Base 2026E': 2.25}
pass_through = 0.50  # conservative: 50bp ECB cut → ~25bp NIM compression

nim_2024 = k24['nim'].to_dict()
ecb_2024 = 4.00

rows = []
for scenario, rate in ecb.items():
    delta_ecb = rate - ecb_2024
    for bank in BANKS:
        nim_scenario = nim_2024[bank] + delta_ecb * pass_through
        rows.append({'Scenario': scenario, 'Bank': bank, 'NIM (%)': round(nim_scenario, 2)})

nim_df = pd.DataFrame(rows)

print('NIM COMPRESSION SENSITIVITY (50% ECB pass-through assumed)')
print(nim_df.pivot(index='Bank', columns='Scenario', values='NIM (%)').loc[BANKS].to_string())
print()
print('Note: A 50bp pass-through means a 150bp ECB cut → ~75bp NIM compression.')
print('NBG (highest NIM) loses the most in absolute terms; Eurobank fee income provides partial offset.')

NIM COMPRESSION SENSITIVITY (50% ECB pass-through assumed)
Scenario      2024A  Base 2025E  Base 2026E  Bear 2025E
Bank                                                   
Eurobank       2.48        1.73        1.60        1.48
Alpha Bank     2.32        1.57        1.44        1.32
Piraeus Bank   2.61        1.86        1.73        1.61
NBG            3.14        2.39        2.27        2.14

Note: A 50bp pass-through means a 150bp ECB cut → ~75bp NIM compression.
NBG (highest NIM) loses the most in absolute terms; Eurobank fee income provides partial offset.


## 4. Sector Risk Matrix

In [4]:
risks = [
    ('ECB rate cuts → NIM compression',       'High',   'High',   'NBG most exposed (NIM 3.14%)'),
    ('Credit cost normalisation',             'Medium', 'Medium', 'Piraeus impairment at 10yr low (€181m)'),
    ('Loan growth slowdown',                  'Medium', 'Medium', 'Volume offset to rate headwind constrained by macro'),
    ('Geopolitical / macro shock',            'Low',    'High',   'Greek sovereign spreads remain volatile'),
    ('Regulatory capital requirements',       'Low',    'Medium', 'CET1 buffers comfortable; Piraeus tightest'),
    ('Fee income compression (digital disrupt)', 'Low', 'Low',  'Banking app competition not yet material in GR'),
    ('M&A integration risk',                  'Medium', 'Medium', 'Eurobank Hellenic consolidation ongoing'),
]

print(f'{"Risk Factor":45} {"Likelihood":12} {"Impact":8} {"Key Exposure"}')
print('-' * 120)
for r in risks:
    print(f'{r[0]:45} {r[1]:12} {r[2]:8} {r[3]}')

print()
print('─── MITIGANTS ───')
mitigants = [
    'All four banks now well above EBA SREP minimum capital requirements',
    'NPL ratios in mid-single digits vs 45%+ at 2016 peak — credit quality structurally improved',
    'Operating leverage: if volumes grow, fixed cost base amplifies profits (C/I 31–38% range)',
    'ECB TLTRO repaid; deposit base sticky and repricing slowly vs asset side',
]
for m in mitigants:
    print(f'  ✓ {m}')

Risk Factor                                   Likelihood   Impact   Key Exposure
------------------------------------------------------------------------------------------------------------------------
ECB rate cuts → NIM compression               High         High     NBG most exposed (NIM 3.14%)
Credit cost normalisation                     Medium       Medium   Piraeus impairment at 10yr low (€181m)
Loan growth slowdown                          Medium       Medium   Volume offset to rate headwind constrained by macro
Geopolitical / macro shock                    Low          High     Greek sovereign spreads remain volatile
Regulatory capital requirements               Low          Medium   CET1 buffers comfortable; Piraeus tightest
Fee income compression (digital disrupt)      Low          Low      Banking app competition not yet material in GR
M&A integration risk                          Medium       Medium   Eurobank Hellenic consolidation ongoing

─── MITIGANTS ───
  ✓ All four 

## 5. ROE vs CET1 — Positioning Chart

In [5]:
fig = go.Figure()

for bank in BANKS:
    row = k24.loc[bank]
    fig.add_trace(go.Scatter(
        x=[row['cet1']], y=[row['roe']],
        mode='markers+text',
        name=bank,
        text=[bank.replace(' Bank', '')],
        textposition='top center',
        textfont=dict(size=11, color=COLORS[bank]),
        marker=dict(
            size=row['total_assets'] / 1000,
            sizemode='area',
            color=COLORS[bank],
            opacity=0.75,
            line=dict(width=1.5, color='white')
        ),
        hovertemplate=(
            f'<b>{bank}</b><br>'
            'CET1: %{x:.1f}%<br>'
            'ROE: %{y:.1f}%<br>'
            f'Assets: €{row["total_assets"]/1000:.0f}bn'
            '<extra></extra>'
        )
    ))

fig.add_vline(x=k24['cet1'].mean(), line_dash='dot', line_color='rgba(255,255,255,0.15)')
fig.add_hline(y=k24['roe'].mean(), line_dash='dot', line_color='rgba(255,255,255,0.15)')

for txt, x, y, anchor in [
    ('High ROE\nHigh Capital', 18.5, 18.0, 'left'),
    ('Low ROE\nHigh Capital',  18.5,  9.5, 'left'),
    ('High ROE\nLow Capital',  13.2, 18.0, 'right'),
    ('Low ROE\nLow Capital',   13.2,  9.5, 'right'),
]:
    fig.add_annotation(x=x, y=y, text=txt, showarrow=False,
                       font=dict(size=8.5, color='rgba(255,255,255,0.18)'),
                       xanchor=anchor)

fig.update_layout(
    title=dict(text='ROE vs CET1 — FY2024 (bubble = assets)', font=dict(size=13), x=0.5),
    paper_bgcolor='#141921', plot_bgcolor='#141921',
    font=dict(family='DM Sans, sans-serif', color='#8892a8', size=11),
    xaxis=dict(title='CET1 Ratio (%)', gridcolor='rgba(255,255,255,0.06)', zeroline=False),
    yaxis=dict(title='ROE (%)', gridcolor='rgba(255,255,255,0.06)', zeroline=False, ticksuffix='%'),
    showlegend=False,
    width=640, height=420,
    margin=dict(l=60, r=40, t=50, b=60)
)
fig.show()
try:
    fig.write_image('../02_Banking_Sector_Dashboard/outputs/roe_cet1_matrix.png', scale=2, width=640, height=420)
except Exception:
    pass

## 6. Sector 3-Year KPI Evolution

In [6]:
metrics = [
    ('roe',            'ROE (%)',       '%'),
    ('nim',            'NIM (%)',       '%'),
    ('cost_to_income', 'C/I Ratio (%)', '%'),
    ('cet1',           'CET1 (%)',      '%'),
]

fig = make_subplots(rows=2, cols=2, subplot_titles=[m[1] for m in metrics],
                    vertical_spacing=0.18, horizontal_spacing=0.12)

for idx, (col, lbl, sfx) in enumerate(metrics):
    r, c = divmod(idx, 2)
    for bank in BANKS:
        bdata = kpis[kpis.bank == bank].sort_values('year')
        fig.add_trace(go.Scatter(
            x=bdata['year'], y=bdata[col],
            name=bank, legendgroup=bank,
            showlegend=(idx == 0),
            mode='lines+markers',
            line=dict(color=COLORS[bank], width=2.2),
            marker=dict(size=6, color=COLORS[bank]),
            hovertemplate=f'{bank}<br>{lbl}: %{{y:.2f}}{sfx}<extra></extra>'
        ), row=r+1, col=c+1)
    fig.update_xaxes(tickvals=[2022, 2023, 2024], row=r+1, col=c+1,
                     gridcolor='rgba(255,255,255,0.06)', zeroline=False)
    fig.update_yaxes(ticksuffix=sfx, row=r+1, col=c+1,
                     gridcolor='rgba(255,255,255,0.06)', zeroline=False)

fig.update_layout(
    paper_bgcolor='#141921', plot_bgcolor='#141921',
    font=dict(family='DM Sans, sans-serif', color='#8892a8', size=10.5),
    title=dict(text='Greek Banks — Key Metrics 2022–2024', font=dict(size=13), x=0.5),
    legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.08, font=dict(size=9.5)),
    width=820, height=520,
    margin=dict(l=55, r=30, t=70, b=60)
)
fig.show()
try:
    fig.write_image('../02_Banking_Sector_Dashboard/outputs/sector_kpi_trends.png', scale=2, width=820, height=520)
except Exception:
    pass

In [7]:
# ── Final validation ──────────────────────────────────────────────────────────
assert len(kpis) == 12, 'Expected 12 rows (4 banks x 3 years)'
assert kpis.bank.nunique() == 4
assert kpis.year.nunique() == 3

# Sector-aggregate sanity: total 2024 NII > 2022 NII (rate cycle tailwind)
nii_22 = kpis[kpis.year == 2022]['nii'].sum()
nii_24 = kpis[kpis.year == 2024]['nii'].sum()
assert nii_24 > nii_22 * 1.4, f'Expected NII to rise >40% 2022-2024, got {nii_24/nii_22-1:.1%}'

# NBG leads on ROA in 2024 (1.54%)
assert k24['roa'].idxmax() == 'NBG', 'NBG should lead on ROA in 2024'

# NBG CET1 should be highest in 2024
assert k24['cet1'].idxmax() == 'NBG', 'NBG should have highest CET1 in 2024'

# Piraeus leads on C/I efficiency (31.8%) — Eurobank second (32.6%)
assert k24['cost_to_income'].idxmin() == 'Piraeus Bank', 'Piraeus should have best C/I in 2024'

print('✅ All checks passed')
print(f'   Sector NII 2022: €{nii_22:,.0f}m  →  2024: €{nii_24:,.0f}m  (+{nii_24/nii_22-1:.1%})')
print(f'   Best ROA 2024:   {k24["roa"].idxmax()} ({k24["roa"].max():.2f}%)')
print(f'   Best CET1 2024:  {k24["cet1"].idxmax()} ({k24["cet1"].max():.1f}%)')
print(f'   Best C/I 2024:   {k24["cost_to_income"].idxmin()} ({k24["cost_to_income"].min():.1f}%)')

✅ All checks passed
   Sector NII 2022: €5,531m  →  2024: €8,594m  (+55.4%)
   Best ROA 2024:   NBG (1.54%)
   Best CET1 2024:  NBG (19.1%)
   Best C/I 2024:   Piraeus Bank (31.8%)
